In [ ]:
import sys
sys.path.append('C:/Users/ab/OneDrive/projects/stocks/src/')

In [ ]:
from stock_downloader.technical_analysis.ta_definitions import (
    talib_functions,
    pattern_columns,
    custom_ta_sets__ma_ratio,
    custom_ta_sets__change_ratio,
    custom_ta_sets__future,
    custom_ta_sets__regression_channel,
    custom_ta_sets__regression_channel_ma,
)

In [ ]:
import duckdb
from pathlib import Path
from pandas import DataFrame

In [ ]:
from stock_downloader.schemas.indicies import indicies_schema
from stock_downloader.schemas.equity_info import equity_info_schema
from stock_downloader.schemas.etf_info import etf_info_schema
from stock_downloader.schemas.price import price_schema
from stock_downloader.schemas.nasdaq_symbols import nasdaq_symbols_schema
from stock_downloader.schemas.other_symbols import other_symbols_schema
from stock_downloader.schemas.talib import talib_schema
from stock_downloader.schemas.ta__change import ta__change_schema
from stock_downloader.schemas.ta__ma_ratio import ta__ma_ratio_schema
from stock_downloader.schemas.regression import regression_schema
from stock_downloader.schemas.regression_indicators import regression_indicators_schema
from stock_downloader.schemas.regression_indicators_ma import regression_indicators_ma_schema
from stock_downloader.schemas.ma_future import ma_future_schema

In [ ]:
results_list = []

In [ ]:
_ = [results_list.append(['talib', list(i.keys())[0], 'function_definitions']) for i in talib_functions]
_ = [results_list.append(['talib', i, 'function_definitions']) for i in pattern_columns]
_ = [results_list.append(['ta__ma_ratio', i.get('output'), 'function_definitions']) for i in custom_ta_sets__ma_ratio]
_ = [results_list.append(['ta__change', i.get('output'), 'function_definitions']) for i in custom_ta_sets__change_ratio]
_ = [results_list.append(['regression_indicators', i.get('output'), 'function_definitions']) for i in custom_ta_sets__regression_channel]
_ = [results_list.append(['regression_indicators_ma', i.get('output'), 'function_definitions']) for i in custom_ta_sets__regression_channel_ma]
_ = [results_list.append(['ma_future', i.get('output'), 'function_definitions']) for i in custom_ta_sets__future]

In [ ]:
_ = [results_list.append(['price', i, 'schemas']) for i in price_schema.columns]
_ = [results_list.append(['talib', i, 'schemas']) for i in talib_schema.columns]
_ = [results_list.append(['ta__ma_ratio', i, 'schemas']) for i in ta__ma_ratio_schema.columns]
_ = [results_list.append(['ta__change', i, 'schemas']) for i in ta__change_schema.columns]
_ = [results_list.append(['regression', i, 'schemas']) for i in regression_schema.columns]
_ = [results_list.append(['regression_indicators', i, 'schemas']) for i in regression_indicators_schema.columns]
_ = [results_list.append(['regression_indicators_ma', i, 'schemas']) for i in regression_indicators_ma_schema.columns]
_ = [results_list.append(['ma_future', i, 'schemas']) for i in ma_future_schema.columns]

In [ ]:
db_folder = Path("D:/stocks/output/database/")
db_folder#.mkdir(exist_ok=True, parents=True)

In [ ]:
db = duckdb.connect(db_folder / "stocks.db")
tables_df = db.execute("SHOW TABLES").df()
metadata_tables_df = db.execute("SELECT * FROM duckdb_tables()").df()

In [ ]:
for table_name in metadata_tables_df.table_name:
    df = db.execute(f"SELECT * FROM {table_name}").df()
    for col in df.columns:
        results_list.append([table_name, col, 'db'])
db.close()

In [ ]:
df = DataFrame(results_list, columns=['table', 'columns', 'source'])

In [ ]:
ta_tables = [
    # 'dowjones',
    # 'equity_info',
    # 'etf_info',
    # 'indicies',
    # 'info',
    'ma_future',
    # 'nasdaq_symbols',
    # 'other_symbols',
    'price',
    'regression',
    'regression_indicators',
    'regression_indicators_ma',
    # 'sp500',
    'talib',
    'ta__change',
    'ta__ma_ratio']

In [ ]:
df_ = df.assign(values=True).pivot_table(index=['table', 'columns'], columns='source', values='values').fillna(False).astype(bool).reset_index(drop=False)
df_ = df_.query('table.isin(@ta_tables)')

In [ ]:
(
    df_
    # .query('schemas == True')
    # .query('db == False')

    # .query('function_definitions == False')
    # .query('schemas == False')
    .query('db == False')
    # .query('table != "talib"')
    .query('columns != "date"')
    .query('columns != "symbol"')
    .query('table != "price"')
    # .query('schemas == False')
    # .tail(30)
)

In [ ]:
df.query('source == "db"').table.drop_duplicates().to_list()